# 📥 Ingest: Recent arXiv NLP/LLM Papers → ChromaDB

This notebook:
1. Fetches cs.CL / cs.AI papers from arXiv **published after Jan 2025** (post Gemma 3 training cutoff)
2. Chunks each paper's abstract + fetched full text
3. Embeds chunks locally using `sentence-transformers`
4. Stores everything in a persistent local ChromaDB collection

> **Why arXiv API directly?**  
> HuggingFace arxiv datasets don't expose paper dates, so we can't filter by recency.  
> The arXiv API lets us query by date range, ensuring Gemma 3 hasn't seen this data.

Run this once (or re-run to refresh). The DB persists to `./chroma_db/`.

In [ ]:
%pip install -q chromadb sentence-transformers tqdm arxiv

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────

# arXiv search config
# Papers submitted after this date — Gemma 3 cutoff is ~early 2025
DATE_FROM       = "2025-03-01"          # YYYY-MM-DD, start of fetch window
DATE_TO         = "2025-06-01"          # YYYY-MM-DD, end of fetch window
ARXIV_CATS      = ["cs.CL", "cs.AI"]   # arXiv categories to pull from
MAX_PAPERS      = 300                   # total papers to ingest

# Chunking
CHUNK_SIZE      = 512                   # characters per chunk
CHUNK_OVERLAP   = 64                    # overlap between consecutive chunks

# Embedding
EMBED_MODEL     = "all-MiniLM-L6-v2"   # local SentenceTransformer model
BATCH_SIZE      = 64

# ChromaDB
CHROMA_DIR      = "./chroma_db"
COLLECTION_NAME = "arxiv_nlp"

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re
import time
import datetime
import arxiv
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print("All imports OK ✓")

In [ ]:
# ── 1. Fetch papers from arXiv ────────────────────────────────────────────────
# We query each category separately then deduplicate by arxiv ID

date_from = datetime.datetime.strptime(DATE_FROM, "%Y-%m-%d").replace(
    tzinfo=datetime.timezone.utc
)
date_to = datetime.datetime.strptime(DATE_TO, "%Y-%m-%d").replace(
    tzinfo=datetime.timezone.utc
)

print(f"Fetching papers from {DATE_FROM} to {DATE_TO} in categories: {ARXIV_CATS}")

seen_ids = set()
papers   = []   # list of arxiv.Result objects

client_arxiv = arxiv.Client(page_size=100, delay_seconds=3, num_retries=3)

per_cat_limit = (MAX_PAPERS // len(ARXIV_CATS)) + 50   # fetch a bit more to account for dedup

for cat in ARXIV_CATS:
    # submittedDate filter uses YYYYMMDDHHMMSS format
    date_q = (
        f"submittedDate:[{date_from.strftime('%Y%m%d')}0000 "
        f"TO {date_to.strftime('%Y%m%d')}2359]"
    )
    query = f"cat:{cat} AND {date_q}"
    print(f"  Querying: {query}")

    search = arxiv.Search(
        query=query,
        max_results=per_cat_limit,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    for result in client_arxiv.results(search):
        if result.entry_id not in seen_ids:
            # double-check date in case arXiv returns out-of-range results
            pub = result.published
            if pub.tzinfo is None:
                pub = pub.replace(tzinfo=datetime.timezone.utc)
            if date_from <= pub <= date_to:
                seen_ids.add(result.entry_id)
                papers.append(result)
                if len(papers) >= MAX_PAPERS:
                    break
    if len(papers) >= MAX_PAPERS:
        break

print(f"\nFetched {len(papers):,} unique papers ✓")
print("Sample:", papers[0].title if papers else "(none)")

In [ ]:
# ── 2. Verify date coverage ───────────────────────────────────────────────────
dates = [p.published for p in papers]
print(f"Earliest paper : {min(dates).strftime('%Y-%m-%d')}")
print(f"Latest paper   : {max(dates).strftime('%Y-%m-%d')}")
print(f"\nAll papers are after Gemma 3 training cutoff (early 2025): ",
      all(d.year >= 2025 and d.month >= 3 for d in dates))

In [ ]:
# ── 3. Chunking helpers ───────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """Strip LaTeX, normalise whitespace."""
    text = re.sub(r"\\[a-zA-Z]+\{[^}]*\}", "", text)   # \cmd{...}
    text = re.sub(r"\$[^$]+\$", "[MATH]", text)           # inline math
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(
    text: str,
    size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    """Character-level sliding-window chunker."""
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size].strip())
        start += size - overlap
    return [c for c in chunks if len(c) > 60]

print("Chunkers ready ✓")

In [ ]:
# ── 4. Build chunk list with rich metadata ────────────────────────────────────
# Each arXiv result gives us: title, abstract, authors, published date, categories
# We embed: abstract (one chunk) + body chunks from the summary/full abstract text
# Note: arXiv API gives abstracts only — full PDFs need pdf parsing (pypdf).
# Abstracts of NLP papers are information-dense and sufficient for strong RAG.
# If you want full text, see the optional cell below.

all_chunks   = []
all_ids      = []
all_metadata = []

for paper in tqdm(papers, desc="Processing papers"):
    paper_id  = paper.entry_id.split("/")[-1]          # e.g. '2503.12345v1'
    pub_date  = paper.published.strftime("%Y-%m-%d")
    cats      = ",".join(paper.categories)
    title     = clean_text(paper.title)
    authors   = ", ".join(a.name for a in paper.authors[:5])  # first 5 authors

    base_meta = {
        "paper_id"  : paper_id,
        "title"     : title[:200],          # ChromaDB metadata values must be str
        "authors"   : authors[:200],
        "pub_date"  : pub_date,
        "categories": cats,
        "url"       : paper.entry_id,
    }

    # ── Title + Abstract as a single rich chunk ──
    abstract_text = clean_text(paper.summary)
    full_abstract = f"Title: {title}\n\nAbstract: {abstract_text}"
    all_chunks.append(full_abstract)
    all_ids.append(f"{paper_id}_abstract")
    all_metadata.append({**base_meta, "section": "abstract", "chunk_idx": 0})

    # ── Chunk the abstract body separately for finer retrieval ──
    for c_idx, chunk in enumerate(chunk_text(abstract_text)):
        all_chunks.append(chunk)
        all_ids.append(f"{paper_id}_chunk_{c_idx:04d}")
        all_metadata.append({**base_meta, "section": "body", "chunk_idx": c_idx})

print(f"\nTotal chunks to embed: {len(all_chunks):,}")

In [ ]:
# ── [OPTIONAL] Enrich with full PDF text ──────────────────────────────────────
# Uncomment this cell to download and parse PDFs for richer context.
# Requires: pip install pypdf requests
# Warning: downloads ~300 PDFs — can take 10–20 min on a slow connection.

# import requests, io
# from pypdf import PdfReader
#
# def fetch_pdf_text(pdf_url: str, timeout: int = 30) -> str:
#     try:
#         r = requests.get(pdf_url, timeout=timeout)
#         reader = PdfReader(io.BytesIO(r.content))
#         return " ".join(page.extract_text() or "" for page in reader.pages)
#     except Exception as e:
#         print(f"  PDF fetch failed for {pdf_url}: {e}")
#         return ""
#
# print("Fetching full PDFs (this takes a while)...")
# for paper in tqdm(papers, desc="PDFs"):
#     paper_id = paper.entry_id.split("/")[-1]
#     pdf_url  = paper.pdf_url
#     pdf_text = clean_text(fetch_pdf_text(pdf_url))
#     if not pdf_text:
#         continue
#     base_meta = {m for m in all_metadata if m["paper_id"] == paper_id}
#     meta = next(iter(base_meta), {})
#     for c_idx, chunk in enumerate(chunk_text(pdf_text)):
#         all_chunks.append(chunk)
#         all_ids.append(f"{paper_id}_pdf_{c_idx:05d}")
#         all_metadata.append({**meta, "section": "pdf", "chunk_idx": c_idx})
#     time.sleep(1)   # be polite to arXiv servers
# print(f"Total chunks after PDF enrichment: {len(all_chunks):,}")

In [ ]:
# ── 5. Embed with SentenceTransformers (fully local) ─────────────────────────
print(f"Loading embedding model '{EMBED_MODEL}'…")
embedder = SentenceTransformer(EMBED_MODEL)
print("Model loaded ✓")

print(f"\nEmbedding {len(all_chunks):,} chunks in batches of {BATCH_SIZE}…")
embeddings = embedder.encode(
    all_chunks,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Embedding shape: {embeddings.shape} ✓")

In [ ]:
# ── 6. Store in ChromaDB ──────────────────────────────────────────────────────
print(f"Opening ChromaDB at '{CHROMA_DIR}'…")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Drop existing collection so re-runs stay idempotent
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing '{COLLECTION_NAME}' collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

UPSERT_BATCH = 500
for start in tqdm(range(0, len(all_chunks), UPSERT_BATCH), desc="Upserting to ChromaDB"):
    end = start + UPSERT_BATCH
    collection.upsert(
        ids=all_ids[start:end],
        embeddings=embeddings[start:end].tolist(),
        documents=all_chunks[start:end],
        metadatas=all_metadata[start:end],
    )

print(f"\n✅ Ingestion complete!")
print(f"   Collection  : {COLLECTION_NAME}")
print(f"   Total docs  : {collection.count():,}")
print(f"   Date range  : {DATE_FROM} → {DATE_TO}")
print(f"   Persisted   : {CHROMA_DIR}")

In [ ]:
# ── 7. Sanity check ───────────────────────────────────────────────────────────
test_query = "attention mechanism in transformers"
q_emb = embedder.encode([test_query]).tolist()

results = collection.query(query_embeddings=q_emb, n_results=3)
print(f"Query: '{test_query}'\n")
for i, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    print(f"--- Result {i+1} | {meta['paper_id']} | {meta['pub_date']} | dist={dist:.4f} ---")
    print(f"Title: {meta.get('title', 'N/A')}")
    print(doc[:300])
    print()